# NLP Assignment 3: Chatbot using Hugging Face Transformers
### Objective
Build a simple conversational chatbot using a pre-trained transformer model (DialoGPT) from Hugging Face that can interact with users and generate meaningful responses.

---
## Step 1: Install Required Libraries

In [ ]:
# Install Hugging Face Transformers and PyTorch (if not already installed)
!pip install transformers torch

---
## Step 2: Import Libraries

In [ ]:
# Import necessary libraries from Hugging Face Transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

---
## Step 3: Load Pre-trained Model and Tokenizer

In [ ]:
# Load DialoGPT-medium — a pre-trained conversational model by Microsoft on Hugging Face
# DialoGPT is specifically fine-tuned for multi-turn dialogue generation

print("Loading model... please wait.")

model_name = "microsoft/DialoGPT-medium"

# Tokenizer converts text to tokens the model understands
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the causal language model for text/response generation
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully!")

---
## Step 4: Define the Response Generation Function

In [ ]:
# Function to generate a chatbot response given user input and conversation history

def generate_response(user_input, chat_history_ids=None):
    """
    Generates a response from DialoGPT based on user input.

    Parameters:
        user_input (str): The message typed by the user.
        chat_history_ids (tensor): Previous conversation context (for multi-turn chat).

    Returns:
        response (str): The chatbot's generated reply.
        chat_history_ids (tensor): Updated conversation history.
    """

    # Tokenize the new user input and add end-of-sequence token
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"  # Return as PyTorch tensor
    )

    # Append new input to chat history for context-aware conversation
    # If no history exists (first message), use only the new input
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate a response using the model
    # max_length limits total conversation length to avoid overflow
    # pad_token_id set to eos_token_id to handle padding properly
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,       # Avoid repeating the same phrases
        do_sample=True,               # Enable sampling for diverse responses
        top_k=50,                     # Consider top 50 tokens at each step
        top_p=0.95,                   # Nucleus sampling for natural responses
        temperature=0.75              # Controls randomness (lower = more focused)
    )

    # Decode only the newly generated tokens (skip the input tokens)
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

---
## Step 5: Run the Chatbot (Interactive Loop)

In [ ]:
# Main chatbot loop — accepts user input and generates responses until 'exit' or 'quit'

print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
print("(Type 'exit' or 'quit' to end the conversation)\n")

chat_history_ids = None  # Start with no conversation history

while True:
    # Accept user input from the console
    user_input = input("You: ").strip()

    # Skip empty inputs — ask user to type something
    if not user_input:
        print("Chatbot: Please type a message.\n")
        continue

    # Exit condition — stop the chatbot when user types exit or quit
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Thank you for chatting! Goodbye! 👋")
        break

    # Generate chatbot response using the transformer model
    response, chat_history_ids = generate_response(user_input, chat_history_ids)

    # Display the generated response
    print(f"Chatbot: {response}\n")

---
## Step 6: Sample Conversation Output
Below is a sample interaction to demonstrate the chatbot working correctly.

In [ ]:
# Demo: Run a fixed set of sample inputs to show chatbot responses
# This cell simulates a conversation without requiring manual input

sample_inputs = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "Thank you"
]

print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

demo_history = None  # Fresh history for demo

for user_msg in sample_inputs:
    print(f"User: {user_msg}")
    reply, demo_history = generate_response(user_msg, demo_history)
    print(f"Chatbot: {reply}\n")

print("User: exit")
print("Chatbot: Thank you for chatting! Goodbye! 👋")

---
## Summary

| Component | Details |
|---|---|
| **Model Used** | `microsoft/DialoGPT-medium` |
| **Library** | Hugging Face Transformers + PyTorch |
| **Conversation Style** | Multi-turn (context-aware) |
| **Exit Condition** | User types `exit` or `quit` |
| **Sampling Strategy** | Top-K (50) + Top-P (0.95) + Temperature (0.75) |

**Pipeline Flow:**
> User Input → Tokenizer → Model (DialoGPT) → Decode Tokens → Display Response → Loop Until Exit